# RAG-Powered Support Assistant for TechNova

This notebook builds a full RAG pipeline in 6 stages:
1. Document Ingestion
2. Chunking
3. Embeddings & Vector Store
4. Retrieval with Metadata Filtering
5. Generation with Source Citations
6. Gradio Chat UI

In [1]:
import os
import json
import random
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)

openai_client = OpenAI()
print("OpenAI API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

OpenAI API key loaded: True


---
## Stage 1 — Document Ingestion

Walk every subfolder of `knowledge-base/` and load each `.md` file as a `Document` object with `source` and `category` metadata.

In [2]:
KB_ROOT = Path("knowledge-base")

docs = []
for md_file in sorted(KB_ROOT.rglob("*.md")):
    category = md_file.parent.name          # subfolder name: products, faqs, policies, etc.
    source = md_file.relative_to(Path(".")).as_posix()  # e.g. knowledge-base/products/nova_phone_x1.md
    text = md_file.read_text(encoding="utf-8")
    docs.append(Document(
        page_content=text,
        metadata={"source": source, "category": category}
    ))

print(f"Loaded {len(docs)} documents")
categories = {}
for d in docs:
    cat = d.metadata["category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} files")

Loaded 34 documents
  company: 4 files
  faqs: 8 files
  policies: 5 files
  products: 6 files
  release_notes: 3 files
  support_tickets: 8 files


In [3]:
# Sanity check — print metadata for 3 random documents
for doc in random.sample(docs, 3):
    print(f"source   : {doc.metadata['source']}")
    print(f"category : {doc.metadata['category']}")
    print(f"length   : {len(doc.page_content)} chars")
    print(f"preview  : {doc.page_content[:120].strip()}")
    print("-" * 60)

source   : knowledge-base/policies/privacy_policy.md
category : policies
length   : 2649 chars
preview  : # Privacy Policy

**Effective Date:** January 1, 2024

## 1. Information We Collect

TechNova collects the following cat
------------------------------------------------------------
source   : knowledge-base/support_tickets/ticket_004.md
category : support_tickets
length   : 980 chars
preview  : # Support Ticket #004

**Status:** Resolved
**Date opened:** 2024-07-08
**Date closed:** 2024-07-09
**Product:** Nova Wa
------------------------------------------------------------
source   : knowledge-base/faqs/software_updates.md
category : faqs
length   : 2063 chars
preview  : # Software Updates FAQ

## How often does TechNova release software updates?
- **Major updates:** Once per year (autumn)
------------------------------------------------------------


---
## Stage 2 — Chunking

Split each document into overlapping chunks (700 chars, 120 overlap). Every chunk inherits `source` and `category` from its parent.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(docs)

# Prepend the source path to every chunk so the embedding model always knows
# which document the chunk belongs to — critical for spec/detail sections that
# don't repeat the product name (e.g. "## Key Specifications" with no title).
for chunk in chunks:
    chunk.page_content = f"[{chunk.metadata['source']}]\n{chunk.page_content}"

print(f"Total chunks: {len(chunks)}")


In [5]:
# Sanity check — confirm metadata survives chunking
for chunk in random.sample(chunks, 2):
    print(f"source   : {chunk.metadata['source']}")
    print(f"category : {chunk.metadata['category']}")
    print(f"length   : {len(chunk.page_content)} chars")
    print(f"content  : {chunk.page_content[:200].strip()}")
    print("-" * 60)

source   : knowledge-base/policies/shipping_policy.md
category : policies
length   : 453 chars
content  : ## 7. P.O. Boxes & APO/FPO Addresses

We can ship small items (earbuds, accessories) to P.O. Boxes. Larger items (laptops, tablets) require a physical street address. APO/FPO addresses are supported v
------------------------------------------------------------
source   : knowledge-base/policies/return_policy.md
category : policies
length   : 678 chars
content  : ## 3. Non-Returnable Items

The following items cannot be returned:

- Earbuds and earphones once the hygienic seal has been broken
- Custom-engraved or personalized products
- Digital products includ
------------------------------------------------------------


---
## Stage 3 — Embeddings & Vector Store

Embed all chunks with `text-embedding-3-small` and persist to `vector_db/`.  
On subsequent runs the existing DB is loaded without re-embedding.

In [ ]:
VECTOR_DB_PATH = "vector_db"
EMBEDDING_MODEL = "text-embedding-3-small"

embedding_fn = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Check for existing Chroma SQLite file to avoid rebuilding every run
chroma_db_file = Path(VECTOR_DB_PATH) / "chroma.sqlite3"

if chroma_db_file.exists():
    print("Loading existing vector store...")
    vectorstore = Chroma(
        persist_directory=VECTOR_DB_PATH,
        embedding_function=embedding_fn
    )
else:
    print(f"Building vector store from {len(chunks)} chunks — calling OpenAI embeddings API...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_fn,
        persist_directory=VECTOR_DB_PATH
    )
    print("Vector store built and persisted to disk.")

count = vectorstore._collection.count()
print(f"Vector store ready — {count} vectors stored")


In [7]:
# Sanity check — pull one record and verify metadata
sample_results = vectorstore.similarity_search("Nova Phone X1 battery", k=1)
s = sample_results[0]
print(f"source   : {s.metadata['source']}")
print(f"category : {s.metadata['category']}")
print(f"content  : {s.page_content[:300].strip()}")

source   : knowledge-base/products/nova_phone_x1.md
category : products
content  : # Nova Phone X1

## Overview
The Nova Phone X1 is TechNova's flagship smartphone, launched in March 2024. It is designed for power users who need long battery life, a premium camera, and fast performance.


---
## Stage 4 — Retrieval with Metadata Filtering

The `retrieve()` function returns the top-k most relevant chunks for a query.  
An optional `category` parameter narrows the search to one subfolder.

In [ ]:
def retrieve(question: str, k: int = 12, category: str = None) -> list:
    """Return top-k relevant Document chunks, optionally filtered by category."""
    search_kwargs = {"k": k}
    if category:
        search_kwargs["filter"] = {"category": category}
    return vectorstore.similarity_search(question, **search_kwargs)


In [9]:
# Sanity check — test several queries including category filter and nonsense
test_queries = [
    ("What is the return window for NovaCare+ members?", "policies"),
    ("How much is the Nova Phone X1?", "products"),
    ("Tell me about the cracked watch ticket", "support_tickets"),
    ("purple banana spaceship", None),
]

for query, cat in test_queries:
    results = retrieve(query, k=3, category=cat)
    print(f"Query  : {query!r}")
    print(f"Filter : {cat}")
    for r in results:
        print(f"  -> {r.metadata['source']}")
    print()

Query  : 'What is the return window for NovaCare+ members?'
Filter : policies
  -> knowledge-base/policies/return_policy.md
  -> knowledge-base/policies/warranty_policy.md
  -> knowledge-base/policies/warranty_policy.md

Query  : 'How much is the Nova Phone X1?'
Filter : products
  -> knowledge-base/products/nova_phone_x1.md
  -> knowledge-base/products/nova_phone_x1.md
  -> knowledge-base/products/nova_smartwatch.md

Query  : 'Tell me about the cracked watch ticket'
Filter : support_tickets
  -> knowledge-base/support_tickets/ticket_004.md
  -> knowledge-base/support_tickets/ticket_004.md
  -> knowledge-base/support_tickets/ticket_007.md

Query  : 'purple banana spaceship'
Filter : None
  -> knowledge-base/products/nova_laptop_pro.md
  -> knowledge-base/products/nova_laptop_pro.md
  -> knowledge-base/company/about.md



---
## Stage 5 — Generation with Source Citations

The `rag_answer()` function:
1. Retrieves the top-5 relevant chunks.
2. Injects them into the system prompt, each labeled with its source file.
3. Instructs the LLM to cite sources inline and refuse out-of-scope questions.

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable support assistant for TechNova, a consumer electronics company.
Your job is to answer questions from the internal support team accurately and completely.

Rules:
1. Answer ONLY using the context provided below. Do not use outside knowledge.
2. After each fact you state, cite the source file in parentheses: (source: policies/return_policy.md)
3. If the context does not contain the answer, say exactly: "I don't have that information in my knowledge base."
4. Never invent facts, prices, dates, or policy terms.
5. Be thorough — include ALL specific numbers, prices, timeframes, model numbers, and key terms from the context. Do not omit important details."""


def rag_answer(question: str, history: list = None) -> str:
    """
    Generate a grounded, cited answer using retrieved context.
    history: list of {"role": ..., "content": ...} dicts in OpenAI format.
    """
    if history is None:
        history = []

    retrieved_chunks = retrieve(question, k=5)

    context_parts = [
        f"[{c.metadata['source']}]\n{c.page_content}"
        for c in retrieved_chunks
    ]
    context = "\n\n---\n\n".join(context_parts)

    system_with_context = (
        SYSTEM_PROMPT
        + "\n\n=== KNOWLEDGE BASE CONTEXT ===\n"
        + context
        + "\n=== END CONTEXT ==="
    )

    messages = [{"role": "system", "content": system_with_context}]
    messages.extend(history)
    messages.append({"role": "user", "content": question})

    response = openai_client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        temperature=0.1,
        max_tokens=800,
    )
    return response.choices[0].message.content


In [11]:
# Test generation: grounded answer, out-of-scope refusal, multi-chunk synthesis
test_questions = [
    "What's the return window for NovaCare+ members?",
    "How do I make pasta?",
    "Tell me about the Nova Phone X1 battery and charging speed.",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("=" * 70)

Q: What's the return window for NovaCare+ members?
A: The return window for NovaCare+ members is extended to 45 days. (source: policies/return_policy.md)
Q: How do I make pasta?
A: I don't have that information in my knowledge base.
Q: Tell me about the Nova Phone X1 battery and charging speed.
A: The Nova Phone X1 is designed for long battery life and fast performance. It comes with a 65W USB-C charger included in the box, which allows for quick charging. Additionally, the device features improved background process management and battery efficiency, especially after the NovaOS 5.1 update, which provides an average 8% battery life improvement on standby (source: knowledge-base/products/nova_phone_x1.md and knowledge-base/release_notes/novaos_5_1.md).


---
## Stage 6 — Gradio Chat UI

Multi-turn chat interface. History is preserved across turns and passed to the LLM.  
The bot cites its sources inline in every answer.

In [16]:
import gradio as gr


def chat_fn(message: str, history: list) -> str:
    # Gradio passes history as list of {"role": ..., "content": ...} dicts when type="messages"
    return rag_answer(message, history=history)


demo = gr.ChatInterface(
    fn=chat_fn,
    title="TechNova Support Assistant",
    description=(
        "Ask me anything about TechNova products, policies, shipping, returns, warranties, and more. "
        "All answers are sourced from the official TechNova knowledge base."
    ),
    examples=[
        "What is the return window for NovaCare+ members?",
        "How much does the Nova Laptop Pro 14 base config cost?",
        "Can I return earbuds after opening them?",
        "What is the phone number for NovaCare+ priority support?",
        "Does TechNova ship to Russia?",
    ],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
